In [0]:
%pip install requests

In [0]:
# Notebook: 1_Bronze_Ingestion
# Language: Python

import requests
import json
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

# Configuration
# NOTE: Processing all 50k+ schemes can be very slow. Limit for demonstration.
SCHEMES_TO_PROCESS = 100

# 1. Get the list of all mutual fund scheme codes
try:
    response = requests.get("https://api.mfapi.in/mf")
    response.raise_for_status()  # Raise an exception for bad status codes
    schemes = response.json()
    print(f"Successfully fetched {len(schemes)} total scheme codes.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching scheme codes: {e}")
    dbutils.notebook.exit("Failed to fetch scheme codes")

In [0]:
# 2. Loop through a subset of schemes to fetch detailed NAV data
all_fund_data = []
for i, scheme in enumerate(schemes[:SCHEMES_TO_PROCESS]):
    scheme_code = scheme['schemeCode']
    print(f"Processing {i+1}/{SCHEMES_TO_PROCESS}: Scheme Code {scheme_code}")
    try:
        fund_response = requests.get(f"https://api.mfapi.in/mf/{scheme_code}")
        fund_response.raise_for_status()
        
        # Add metadata to the raw response
        fund_data = {
            "scheme_code": str(scheme_code),
            "raw_data": fund_response.text, # Store the raw JSON as a string
            "ingestion_timestamp": current_timestamp()
        }
        all_fund_data.append(fund_data)
    except requests.exceptions.RequestException as e:
        print(f"Could not fetch data for scheme {scheme_code}: {e}")

In [0]:
# 3. Create a DataFrame and write to the Bronze Delta table
BRONZE_TABLE = "mutual_fund_project.bronze.raw_fund_data"
if not all_fund_data:
    print("No data was fetched. Exiting.")
    dbutils.notebook.exit("No data fetched")

# Define schema for consistency
schema = StructType([
    StructField("scheme_code", StringType(), True),
    StructField("raw_data", StringType(), True)
])

# Convert list of dicts to DataFrame
raw_df = spark.createDataFrame(data=[(d['scheme_code'], d['raw_data']) for d in all_fund_data], schema=schema)
final_df = raw_df.withColumn("ingestion_timestamp", current_timestamp())

# Overwrite the bronze table with the latest full pull
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE_TABLE)

print(f"Successfully ingested data for {final_df.count()} schemes into {BRONZE_TABLE}")

In [0]:
%sql
 select * from mutual_fund_project.bronze.raw_fund_data limit 100;